# Week 4 Assignment: Predictive Modeling in Logistics

In this notebook, I will use shipment details to predict how many days a delivery may take. I will try three models, compare their results, and suggest a few ways a logistics team could use the predictions.

**Note:** The dataset is made for practice. It is not real company data, so the results are only an example.

## 1. Import the libraries

I will use pandas and NumPy to work with data, scikit-learn to build the models, and Matplotlib to make charts. If any library is missing, install it with `pip install pandas numpy scikit-learn matplotlib`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully.")

## 2. Make a sample dataset

Each row represents one shipment. The model will try to predict `Delivery_Time_Days` using distance, shipment volume, region, transport mode, and priority.

The values are randomly generated for this assignment, so they should not be treated as real delivery records.

In [ ]:
n_rows = 600

regions = np.random.choice(
    ["North", "South", "East", "West", "Central"],
    size=n_rows,
    p=[0.22, 0.20, 0.18, 0.20, 0.20]
)
transport_modes = np.random.choice(
    ["Road", "Rail", "Air"],
    size=n_rows,
    p=[0.65, 0.25, 0.10]
)
priorities = np.random.choice(
    ["Standard", "Express", "Urgent"],
    size=n_rows,
    p=[0.60, 0.30, 0.10]
)

distance_km = np.random.randint(50, 2001, size=n_rows)
shipment_volume = np.random.randint(1, 101, size=n_rows)

mode_days = pd.Series(transport_modes).map({"Road": 1.5, "Rail": 2.0, "Air": 0.4}).to_numpy()
priority_days = pd.Series(priorities).map({"Standard": 1.0, "Express": 0.2, "Urgent": -0.4}).to_numpy()
region_days = pd.Series(regions).map({
    "North": 0.2, "South": 0.4, "East": 0.7, "West": 0.1, "Central": 0.3
}).to_numpy()

noise = np.random.normal(loc=0, scale=0.7, size=n_rows)

delivery_time_days = (
    1.2
    + distance_km / 450
    + shipment_volume / 120
    + mode_days
    + priority_days
    + region_days
    + noise
)
delivery_time_days = np.maximum(delivery_time_days, 0.5).round(2)

df = pd.DataFrame({
    "Distance_KM": distance_km,
    "Shipment_Volume": shipment_volume,
    "Region": regions,
    "Transport_Mode": transport_modes,
    "Priority": priorities,
    "Delivery_Time_Days": delivery_time_days
})

print("Dataset shape:", df.shape)
display(df.head())

## 3. Check and prepare the data

First, I will check the columns and look for missing values. Then I will separate the information used for prediction (`X`) from the value I want to predict (`y`). The text columns, such as region and transport mode, will be converted into a format the models can use.

In [ ]:
print("Data types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nSummary statistics:")
display(df.describe(include="all"))

In [ ]:
target = "Delivery_Time_Days"
X = df.drop(columns=[target])
y = df[target]

numeric_features = ["Distance_KM", "Shipment_Volume"]
categorical_features = ["Region", "Transport_Mode", "Priority"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 4. Train three models

I will compare Linear Regression, Decision Tree, and Random Forest. The data is split into training and testing parts. The models learn from the training data, and the test data is used to check their predictions.

In [ ]:
def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
            ("numeric", "passthrough", numeric_features)
        ]
    )

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=6, random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(
        n_estimators=200, max_depth=8, random_state=RANDOM_STATE, n_jobs=-1
    )
}

trained_models = {}
results = []

for name, estimator in models.items():
    pipeline = Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", estimator)
    ])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    trained_models[name] = pipeline
    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

results_df = pd.DataFrame(results).sort_values("MAE")
display(results_df.round(3))

### What do the scores mean?

- **MAE:** The average size of the prediction error, in days. A lower value is better.
- **RMSE:** Similar to MAE, but it gives more weight to larger errors. A lower value is better.
- **R²:** Shows how well the model explains changes in delivery time. Values closer to 1 are generally better.

I will compare all three scores instead of judging a model by only one number.

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(results_df["Model"], results_df["MAE"])
plt.title("Model Comparison by Mean Absolute Error")
plt.xlabel("Model")
plt.ylabel("MAE (days)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
best_predictions = best_model.predict(X_test)

print("Best model by test MAE:", best_model_name)

plt.figure(figsize=(6, 6))
plt.scatter(y_test, best_predictions, alpha=0.65)
min_value = min(y_test.min(), best_predictions.min())
max_value = max(y_test.max(), best_predictions.max())
plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")
plt.title(f"Actual vs Predicted Delivery Time: {best_model_name}")
plt.xlabel("Actual delivery time (days)")
plt.ylabel("Predicted delivery time (days)")
plt.tight_layout()
plt.show()

## 5. Cross-validation

One train-test split may not tell the whole story. Cross-validation repeats the evaluation across five different splits of the training data. I will calculate the average MAE to see how consistent the model is.

In [ ]:
cv_pipeline = trained_models[best_model_name]
cv_scores = cross_val_score(
    cv_pipeline,
    X_train,
    y_train,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)
cv_mae = -cv_scores

print("Five-fold CV MAE values:", np.round(cv_mae, 3))
print("Mean CV MAE:", round(cv_mae.mean(), 3), "days")
print("Standard deviation:", round(cv_mae.std(), 3), "days")

## 6. Tune the Random Forest model

A model has settings that can affect its results. These are called hyperparameters. Here, GridSearchCV tries a few values for tree depth and minimum leaf size, then uses five-fold cross-validation to find a setting with a low MAE.

In [ ]:
random_forest_pipeline = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("model", RandomForestRegressor(
        n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1
    ))
])

param_grid = {
    "model__max_depth": [4, 6, 8, None],
    "model__min_samples_leaf": [1, 2, 4]
}

grid_search = GridSearchCV(
    estimator=random_forest_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

tuned_model = grid_search.best_estimator_
tuned_predictions = tuned_model.predict(X_test)

print("Best parameters:", grid_search.best_params_)
print("Best mean CV MAE:", round(-grid_search.best_score_, 3), "days")
print("Tuned test MAE:", round(mean_absolute_error(y_test, tuned_predictions), 3), "days")
print("Tuned test RMSE:", round(np.sqrt(mean_squared_error(y_test, tuned_predictions)), 3), "days")
print("Tuned test R²:", round(r2_score(y_test, tuned_predictions), 3))

## 7. Review predicted delivery times

The table below shows shipments with the highest predicted delivery times. A logistics team could review these shipments and check whether a delay is likely.

This is only an example. A real route-planning system would also need information such as traffic, road conditions, vehicle capacity, and delivery deadlines.

In [ ]:
review = X_test.copy()
review["Actual_Delivery_Time_Days"] = y_test
review["Predicted_Delivery_Time_Days"] = tuned_predictions
review["Prediction_Error_Days"] = (
    review["Actual_Delivery_Time_Days"] - review["Predicted_Delivery_Time_Days"]
)
review["Review_Flag"] = np.where(
    review["Predicted_Delivery_Time_Days"] >= review["Predicted_Delivery_Time_Days"].quantile(0.75),
    "Review for possible delay",
    "Normal review"
)

display(
    review.sort_values("Predicted_Delivery_Time_Days", ascending=False).head(10).round(2)
)

## 8. Recommendations

Based on this example, a logistics team could:

1. **Check shipments with longer predicted delivery times** and update customers if needed.
2. **Compare transport options** using both delivery time and cost before changing a shipment's mode.
3. **Plan staff and loading capacity** using shipment volume.
4. **Compare regions** to find places where deliveries often take longer.
5. **Balance speed and cost** rather than always choosing the fastest option.
6. **Check model accuracy over time** by comparing predictions with real delivery results.

These suggestions should be tested with real shipment data before being used for actual business decisions.

## 9. Limitations

- The data is simulated for this assignment, not taken from a real company.
- Real delivery times can also depend on traffic, weather, holidays, warehouse delays, and failed delivery attempts.
- The model predicts delivery time; it does not automatically calculate the cheapest or shortest route.
- The scores may change if the generated data changes.

For these reasons, the results should be treated as a learning example.

## 10. Conclusion

In this assignment, I created a sample logistics dataset, trained three models, and compared their predictions using MAE, RMSE, and R². I also used cross-validation and hyperparameter tuning. The predictions can help highlight shipments that may need extra attention, but real company data would be needed before making operational decisions.

The final section contains the code used to create the sample dataset.